# §3.3 + §4.1 — Shape Metrics & Cluster Characterisation

Computes all 9 spatial metrics per cluster and produces the descriptive analysis.

**Inputs:** `data/filtered_clusters.csv`  
**Outputs:** `data/clusters_with_metrics.csv`, Figures 2, 3, 4

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from clustering import cluster_nnd_stats
from metrics.spatial import clark_evans_index
from metrics.topology import euler_number

In [ ]:
df = pd.read_csv('../data/filtered_clusters.csv')
print(f'{len(df):,} clusters loaded')
df.head()

## Compute remaining metrics (Clark-Evans, NND stats, Euler number)

In [ ]:
# filtered_clusters.csv already contains shape metrics from notebook 01.
# Here we reload the original tree-level data to compute per-cluster NND and Clark-Evans.
trees = pd.read_csv('../data/clusters_with_distance.csv')
trees = trees[trees.cluster_id.isin(df.cluster_id)]

ce_list, nnd_mean_list, nnd_std_list, nnd_skew_list, euler_list = [], [], [], [], []

for cid, grp in trees.groupby('cluster_id'):
    pts = grp[['x', 'y']].values
    # NND statistics
    stats = cluster_nnd_stats(pts)
    nnd_mean_list.append(stats['nnd_mean'])
    nnd_std_list.append(stats['nnd_std'])
    nnd_skew_list.append(stats['nnd_skew'])
    # Clark-Evans (area from cluster-level metrics)
    area = df.loc[df.cluster_id == cid, 'area_m2'].values[0]
    ce_list.append(clark_evans_index(pts, area))
    # Euler number
    euler_list.append(euler_number(pts))

df['nnd_mean']     = nnd_mean_list
df['nnd_std']      = nnd_std_list
df['nnd_skew']     = nnd_skew_list
df['clark_evans']  = ce_list
df['euler_number'] = euler_list

print(f"Clark-Evans mean: {df['clark_evans'].mean():.2f}  (expect ~2.18)")
df[['clark_evans','nnd_mean','euler_number']].describe()

In [ ]:
df.to_csv('../data/clusters_with_metrics.csv', index=False)
print('Saved → data/clusters_with_metrics.csv')

## Figure 2 — Histograms of key metrics

In [ ]:
plot_cols = [
    ('n_points',        'Number of trees'),
    ('area_m2',         'Area (m²)'),
    ('perimeter_m',     'Perimeter (m)'),
    ('compactness',     'Form factor'),
    ('aspect_ratio',    'Elongation ratio'),
    ('fractal_dim',     'Fractal dimension'),
    ('clark_evans',     'Clark-Evans index'),
    ('alpha_compactness','Alpha compactness'),
]

fig, axes = plt.subplots(2, 4, figsize=(16, 7), dpi=150)
for ax, (col, label) in zip(axes.flat, plot_cols):
    data = df[col].dropna()
    ax.hist(data, bins=40, color='#4477AA', edgecolor='white', linewidth=0.4)
    ax.set_title(f'({chr(97 + plot_cols.index((col,label)))})', fontsize=10, loc='left')
    ax.set_xlabel(label, fontsize=9)
    ax.set_ylabel('Count', fontsize=9)

plt.suptitle('Distribution of cluster metrics', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../figures/fig2_histograms.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 3 — Correlation matrix heatmap

In [ ]:
corr_cols = ['n_points','area_m2','perimeter_m','compactness',
             'aspect_ratio','fractal_dim','clark_evans','alpha_compactness']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7), dpi=150)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title('Correlation matrix of cluster metrics', fontsize=13)
plt.tight_layout()
plt.savefig('../figures/fig3_correlation.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 4 — Key scatter plots

In [ ]:
pairs = [
    ('clark_evans',     'fractal_dim',      'Clark-Evans index',   'Fractal dimension'),
    ('clark_evans',     'aspect_ratio',     'Clark-Evans index',   'Elongation'),
    ('fractal_dim',     'alpha_compactness','Fractal dimension',   'Alpha compactness'),
    ('fractal_dim',     'compactness',      'Fractal dimension',   'Form factor'),
    ('fractal_dim',     'perimeter_m',      'Fractal dimension',   'Perimeter (m)'),
    ('alpha_compactness','perimeter_m',     'Alpha compactness',   'Perimeter (m)'),
]

fig, axes = plt.subplots(2, 3, figsize=(14, 8), dpi=150)
for ax, (xc, yc, xl, yl) in zip(axes.flat, pairs):
    sub = df[[xc, yc]].dropna()
    ax.scatter(sub[xc], sub[yc], s=3, alpha=0.15, color='#4477AA')
    r = sub.corr().iloc[0, 1]
    ax.set_xlabel(xl, fontsize=9)
    ax.set_ylabel(yl, fontsize=9)
    ax.set_title(f'r = {r:.2f}', fontsize=9)

plt.suptitle('Key metric relationships', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../figures/fig4_scatters.png', dpi=300, bbox_inches='tight')
plt.show()